# Lab 10

> Not adapted from an existing lab: a fresh implementation for this course, following the
> tool chain planned for Session 13 (sentence embeddings from Hugging Face -> TensorBoard
> projector -> MLflow tracking -> the same runs in Weights & Biases). The mean-pooling
> recipe for `sentence-transformers/all-MiniLM-L6-v2` follows that model's card on the
> Hugging Face Hub (Apache 2.0).

Training a model is only part of the work. You also want to *look* at what a model has
learned, and to keep track of which run used which settings and gave which result -
otherwise you cannot reproduce or compare your experiments a week later. This lab walks
through three widely used tools for that, on one small task:

1. embed news headlines with a small pretrained Transformer (MiniLM),
2. explore the embeddings visually in **TensorBoard**'s embedding projector,
3. train classifiers on the embeddings and track every run with **MLflow**,
4. (optional) log the same runs to **Weights & Biases** (W&B).

TensorBoard and MLflow run entirely on your machine (or Colab runtime): no account,
nothing leaves the computer. W&B is a hosted service; the optional exercise runs it in
offline mode by default.

> **Instructor timing (2h session).** ~10 min: intro and setup (the model and dataset
> downloads take a minute). ~25 min: Exercise 1 - mean pooling, embed the data (about a
> minute on CPU). ~20 min: Exercise 2 - write the embeddings for TensorBoard and explore
> them in the projector. ~40 min: Exercise 3 - the tracked training function, a small
> hyperparameter grid, comparing runs in MLflow. ~15 min: Exercise 4 (optional) - W&B.
> ~10 min buffer and discussion.

## Setup

These labs are designed to run on [Google Colab](https://colab.research.google.com/) -
no local Python install needed. Open this notebook via Colab's GitHub loader
(`colab.research.google.com/github/KarikS/lecture_advanced_ai_ml/blob/main/<path-to-this-notebook>`)
or File > Open notebook > GitHub tab, repo `KarikS/lecture_advanced_ai_ml`, then run
the cell below once per session. Running the notebook locally instead (e.g. via the
`exercises/python/` venv described in that folder's README) works unchanged - the cell
below is a no-op there. No GPU is needed.

In [1]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import os
    if not os.path.exists('lecture_advanced_ai_ml'):
        !git clone https://github.com/KarikS/lecture_advanced_ai_ml.git
    !pip install -q mlflow

## Imports

In [2]:
import os
import time

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import torch
import torch.nn.functional as F
from datasets import load_dataset
from matplotlib_inline.backend_inline import set_matplotlib_formats
from torch import nn, Tensor
from torch.utils.tensorboard import SummaryWriter
from transformers import AutoModel, AutoTokenizer

set_matplotlib_formats('png', 'pdf')
torch.manual_seed(0)

## Exercise 1: Sentence Embeddings with MiniLM

[AG News](https://huggingface.co/datasets/fancyzhx/ag_news) contains news articles
(title and first sentence) in four categories: World, Sports, Business, Sci/Tech. We use
a small random subset: 1,000 articles for training and 500 for testing.

In [3]:
ag_news = load_dataset('fancyzhx/ag_news')
label_names = ag_news['train'].features['label'].names
train_data = ag_news['train'].shuffle(seed=0).select(range(1000))
test_data = ag_news['test'].shuffle(seed=0).select(range(500))
print(label_names)
for example in train_data.select(range(3)):
    print(f"[{label_names[example['label']]}] {example['text'][:100]}...")

['World', 'Sports', 'Business', 'Sci/Tech']
[World] India Warns U.S. on Arms Sales to Pakistan  WASHINGTON (Reuters) - India warned on Friday that new  ...
[Business] WTO rules against US, EU on sugar, cotton Latin America #39;s agricultural giant scored two trade vi...
[Sports] Braves' Thomson Leaves Game 3 (AP) AP - Atlanta Braves starter John Thomson reaggravated a sore musc...


[`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) is a
6-layer Transformer encoder (22M parameters), trained so that texts with similar meaning
get similar vectors. Like BERT (Session 8), it outputs one 384-dimensional vector per
token. To get **one** vector per text, the model's card prescribes **mean pooling**:
average the token vectors, but only over real tokens - padding tokens (attention mask
0) must not count. Then normalise the result to unit length, so that the dot product of
two embeddings is their cosine similarity.

In [4]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(model_name)
encoder = AutoModel.from_pretrained(model_name).eval()


def mean_pooling(token_embeddings: Tensor, attention_mask: Tensor) -> Tensor:
    """token_embeddings: (batch, seq_len, dim), attention_mask: (batch, seq_len) of 0/1."""
# TODO: Multiply the token embeddings by the mask (broadcast it to the last
# dimension), sum over the sequence dimension, and divide by the number of real
# tokens per text (clamp it to at least 1e-9 to avoid dividing by 0).


@torch.no_grad()
def embed(texts: list[str], batch_size: int = 64) -> Tensor:
    chunks = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                          max_length=128, return_tensors='pt')
        token_embeddings = encoder(**batch).last_hidden_state
        sentence_embeddings = mean_pooling(token_embeddings, batch['attention_mask'])
        chunks.append(F.normalize(sentence_embeddings, p=2, dim=1))
    return torch.cat(chunks)

In [5]:
# Sanity check: padding must not change a text's embedding.
alone = embed(['The match ended in a draw.'])
padded = embed(['The match ended in a draw.', 'A much longer second sentence forces padding of the first one.'])[:1]
print(torch.allclose(alone, padded, atol=1e-5))
assert torch.allclose(alone, padded, atol=1e-5)

In [6]:
sentences = ['The striker scored twice in the final.',
             'The team won the championship game.',
             'The central bank raised interest rates.',
             'Stock markets fell after the announcement.']
e = embed(sentences)
print((e @ e.T).numpy().round(2))

The two sports sentences should be more similar to each other than to the two business
sentences, even though they share hardly any words. Now embed the whole subset.

In [7]:
start = time.time()
X_train = embed(train_data['text'])
X_test = embed(test_data['text'])
y_train = torch.tensor(train_data['label'])
y_test = torch.tensor(test_data['label'])
print(X_train.shape, X_test.shape, f'{time.time() - start:.0f}s')

**Question:** Why would plain averaging over *all* positions, padding included, be a bug -
and why would you not notice it on a single sentence?

## Exercise 2: Looking at the Embeddings in TensorBoard

TensorBoard's **projector** shows high-dimensional vectors in 2D or 3D, using PCA, t-SNE
or UMAP, and can colour every point by a label. `SummaryWriter.add_embedding` writes the
vectors and one metadata label per vector into a log directory.

In [8]:
log_dir = 'runs/lab10'
writer = SummaryWriter(log_dir)
# TODO: Write the test embeddings with writer.add_embedding, passing the category
# name of every test example as metadata and tag='ag_news_minilm'.
writer.close()
print(sorted(os.listdir(log_dir)))

Now start TensorBoard and open the **Projector** tab (on the top right, it may be hidden
under "Inactive" in the drop-down).

- **Colab:** run the next cell; TensorBoard opens inside the notebook.
- **Locally:** in a terminal, inside `exercises/python/`, run
  `tensorboard --logdir runs` and open `http://localhost:6006`.

In the projector, choose "Color by: label". Try PCA first, then t-SNE (let it run for a
few hundred iterations).

> If the Projector tab stays empty in Colab, download the `runs/` folder (Files panel on
> the left) and open it with a local TensorBoard instead.

In [9]:
if IN_COLAB:
    %load_ext tensorboard
    %tensorboard --logdir runs

**Question:** Do the four categories form separate clusters? Which two overlap most, and
does that make sense?

## Exercise 3: Tracking Experiments with MLflow

We now train classifiers on the frozen embeddings - a linear layer, and a small MLP - with
a few learning rates. That is already 4 runs, each with its own settings, curves and
final result. MLflow records all of this in a local database:

- `mlflow.set_tracking_uri('sqlite:///mlflow.db')` stores everything in a SQLite file in
  the current folder.
- `mlflow.set_experiment(name)` groups runs.
- Inside `with mlflow.start_run(run_name=...):`, `mlflow.log_params(dict)` records the
  settings, and `mlflow.log_metric(name, value, step=...)` records a value per epoch.

In [10]:
mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('lab10-agnews')


def make_classifier(hidden: int) -> nn.Module:
    if hidden == 0:
        return nn.Linear(384, 4)
    return nn.Sequential(nn.Linear(384, hidden), nn.ReLU(), nn.Linear(hidden, 4))


def train_tracked(lr: float, hidden: int, epochs: int = 20, batch_size: int = 64,
                  seed: int = 0) -> float:
    torch.manual_seed(seed)
    model = make_classifier(hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    with mlflow.start_run(run_name=f'lr={lr} hidden={hidden}'):
# TODO: Log the hyperparameters (lr, hidden, epochs, batch_size, seed) and the
# training set size as params.
        for epoch in range(epochs):
            model.train()
            permutation = torch.randperm(len(y_train))
            for i in range(0, len(y_train), batch_size):
                idx = permutation[i:i + batch_size]
                loss = F.cross_entropy(model(X_train[idx]), y_train[idx])
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            model.eval()
            with torch.no_grad():
                train_loss = F.cross_entropy(model(X_train), y_train).item()
                test_acc = (model(X_test).argmax(dim=1) == y_test).float().mean().item()
# TODO: Log train_loss and test_acc for this epoch (step=epoch).
        mlflow.log_metric('final_test_acc', test_acc)
    return test_acc

In [11]:
for lr in (1e-3, 1e-2):
    for hidden in (0, 128):
        acc = train_tracked(lr, hidden)
        print(f'lr={lr:<6} hidden={hidden:<4} final test accuracy={acc:.3f}')

All four runs are now in `mlflow.db`. You can query them from Python - this works the
same in Colab and locally:

In [12]:
runs = mlflow.search_runs(experiment_names=['lab10-agnews'], order_by=['metrics.final_test_acc DESC'])
runs[['tags.mlflow.runName', 'params.lr', 'params.hidden', 'metrics.final_test_acc']]

In [13]:
client = mlflow.tracking.MlflowClient()
plt.figure(figsize=(7, 4))
for _, run in runs.iterrows():
# TODO: Fetch the test_acc history of this run with client.get_metric_history and
# plot it against the step, labelled with the run name.
plt.xlabel('epoch')
plt.ylabel('test accuracy')
plt.legend()
plt.show()

**Locally**, MLflow also has a web UI: in a terminal, inside `exercises/python/`, run
`mlflow ui --backend-store-uri sqlite:///mlflow.db` and open `http://localhost:5000`.
Select the four runs and click "Compare" to see their parameters side by side and their
curves in one plot. (In Colab the UI is not reachable without extra tunnelling, which is
why the cells above query the runs from Python.)

**Question:** These runs log the hyperparameters. What else would you need to log to
reproduce the best run exactly, a month from now?

**Question:** The best runs differ by about one or two percentage points. Is that a real
difference?

## Exercise 4 (optional): The Same Runs in Weights & Biases

W&B does the same job as MLflow, but as a hosted service: runs are uploaded to
`wandb.ai`, where you get dashboards and can share them with a team. Students and
academics can apply for a free academic license (sign up with your university email).

The cell below runs W&B in **offline** mode: it logs to a local `wandb/` folder and needs
no account. To upload, create an account, run `wandb login` with the API key from your
account settings, set `WANDB_MODE` to `'online'` below, and re-run - or upload the offline
runs later with `wandb sync wandb/offline-run-*`.

In [14]:
try:
    import wandb
except ImportError:
    %pip install -q wandb
    import wandb

os.environ['WANDB_MODE'] = 'offline'
# Keep W&B's end-of-run summary out of the notebook output.
os.environ['WANDB_SILENT'] = 'true'

In [15]:
def train_wandb(lr: float, hidden: int, epochs: int = 20, batch_size: int = 64,
                seed: int = 0) -> float:
    torch.manual_seed(seed)
    model = make_classifier(hidden)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    config = {'lr': lr, 'hidden': hidden, 'epochs': epochs, 'batch_size': batch_size,
              'seed': seed, 'n_train': len(y_train)}
# TODO: Start a run with wandb.init(project='lab10-agnews', config=config,
# name=...), and replace the MLflow logging by run.log({...}, step=epoch).
# Finish the run at the end.
    return test_acc


for lr in (1e-3, 1e-2):
    for hidden in (0, 128):
        train_wandb(lr, hidden)

**Question:** TensorBoard, MLflow, W&B - when would you pick which?

## Conclusion

Here's what you should take away from this lab:

 - A pretrained sentence encoder plus mean pooling (Exercise 1) turns texts into vectors
in a few lines; the masking detail matters as soon as you embed batches.
 - Looking at representations (Exercise 2) is cheap with TensorBoard's projector and
often tells you quickly whether a representation separates what you care about.
 - Experiment tracking (Exercise 3) is a habit, not a library: log every setting that
can change the result, log curves rather than only the final number, and compare runs
from the record instead of from memory. MLflow does this locally with a single SQLite
file; W&B (Exercise 4) does the same as a hosted service.